# RUNTIME

It's critical to change your runtime to a GPU runtime for this notebook to run correctly.

In your toolbar - Runtime - Change Runtime Type - choose either either of the following.

T4 GPU

v5e-1 TPU


> ### Note on Labs and Assignments
>
> 🔧 Look for the **wrench emoji** — it marks code you must change. Routine run-only cells do not use it.
>
> 🖊 Look for the **writing emoji** — it marks analysis you must write.
>
> These sections are graded and are not optional.


# Module 1 Lab 1: AI Project Classification and Decision Boundaries

**Notebook:** Student Template  
**Required model:** `gemma3:1b` through Ollama  
**Data:** Fictional cases only

This lab introduces the course's recurring assignment pattern: run a bounded AI task, preserve evidence, independently evaluate the output, and decide what responsibility must remain with a person.


## Learning Objectives

By completing this notebook, you will:

1. Classify business projects by their primary AI approach.
2. Evaluate model reasoning rather than treating it as an answer key.
3. Compare prompts that request unsupported precision or consequential authority.
4. Redesign AI use from decision making to evidence gathering.
5. Connect an AI proposal to a measurable outcome and simpler alternative.


## Important Instructions

1. Read the assignment and Module 1 reading first.
2. Use the fixed `gemma3:1b` model; do not substitute another model.
3. Change code where you see `🔧` and write analysis where you see `🖊`.
4. Run cells from top to bottom and preserve every output.
5. Use only the supplied fictional cases and résumé.
6. Restart the kernel and run all cells before submitting.

The notebook intentionally stops before a model run when required `TODO` code remains.


## Setup Ollama

Run the next two cells. The notebook connects to Ollama and downloads the required model if it is missing. The one-time `gemma3:1b` download is approximately 815 MB.


In [13]:
import subprocess
import time

In [14]:
# Download and install Ollama (Google Colab only — skip if running locally)
install_zstd = subprocess.run(
    "sudo apt-get install zstd",
    shell=True,
    capture_output=True,
    text=True,
)

install_zstd

CompletedProcess(args='sudo apt-get install zstd', returncode=0, stdout='Reading package lists...\nBuilding dependency tree...\nReading state information...\nzstd is already the newest version (1.4.8+dfsg-3build1).\n0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.\n', stderr='')

In [15]:
# RUN THIS CELL. YOU SHOULD SEE Ollama installed and Ollama server is running messages.

# Download and install Ollama
install = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    capture_output=True,
    text=True,
)
if install.returncode != 0:
    raise RuntimeError(f"Ollama installation failed:\n{install.stderr}")
print("Ollama installed.")

# Start the Ollama server as a background process
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Give the server a few seconds to initialize before any requests are made
time.sleep(3)
print("Ollama server is running.")

Ollama installed.
Ollama server is running.


In [16]:
# RUN THIS CELL
from datetime import datetime
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json
import textwrap

from IPython.display import Markdown, display

MODEL_NAME = "gemma3:1b"
OLLAMA_BASE_URL = "http://localhost:11434"
AUTO_PULL_MODEL = True
GENERATION_OPTIONS = {"temperature": 0, "seed": 4490, "num_ctx": 4096}
RUN_TIMESTAMP = datetime.now().astimezone().isoformat(timespec="seconds")


print(f"Required model: {MODEL_NAME}")
print(f"Run date: {RUN_TIMESTAMP}")


Required model: gemma3:1b
Run date: 2026-08-27T22:26:25+00:00


In [17]:
# RUN THIS CELL
def require_finished(label, value):
    if value is None or "TODO" in str(value) or str(value).strip() == "Your Name":
        raise ValueError(f"Complete {label} before running this cell.")


def ollama_request(path, payload=None, timeout=120):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        f"{OLLAMA_BASE_URL}{path}",
        data=data,
        headers={"Content-Type": "application/json"},
        method="GET" if payload is None else "POST",
    )
    try:
        with urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        details = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama returned HTTP {exc.code}: {details}") from exc
    except URLError as exc:
        raise RuntimeError(
            "Cannot connect to Ollama at http://localhost:11434. "
            "Install and start Ollama, then rerun this cell."
        ) from exc


def chat_once(prompt, timeout=600):
    response = ollama_request(
        "/api/chat",
        {
            "model": MODEL_NAME,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "options": GENERATION_OPTIONS,
        },
        timeout=timeout,
    )
    return response["message"]["content"].strip()


version_info = ollama_request("/api/version")
available_models = {
    item.get("name") or item.get("model")
    for item in ollama_request("/api/tags").get("models", [])
}
print(f"Connected to Ollama {version_info.get('version', 'unknown version')}.")

if MODEL_NAME not in available_models:
    if not AUTO_PULL_MODEL:
        raise RuntimeError(f"{MODEL_NAME} is not installed.")
    print(f"Downloading {MODEL_NAME}. This is a one-time download...")
    ollama_request("/api/pull", {"model": MODEL_NAME, "stream": False}, timeout=3600)

print(f"{MODEL_NAME} is ready.")


Connected to Ollama 0.33.1.
gemma3:1b is ready.


# Part 1: Classify Four AI Projects

Write one prompt that covers all four fictional retailer cases. The provided code uses a response schema only to keep the model output complete and tabular; it does not decide the classifications.


### TODO - INSTRUCT 🔧


In [18]:
# 🔧 TODO - INSTRUCT
# Replace the TODO text with your own prompt. Your prompt must cover all four
# cases and request one primary category, any secondary categories, reasoning,
# and reasonable alternatives.
student_prompt = """

You are an expert AI taxonomy analyst. Review the four cases provided below and complete the following steps for each individual case.

For each case:
1. Select exactly one primary AI category. Do not combine categories or list multiples for the primary slot.
2. Identify any important secondary AI categories that apply.
3. Explain your reasoning clearly for both the primary and secondary classifications based on the case details.
4. Acknowledge reasonable alternative categories when classifications overlap, explaining why they are plausible but ultimately secondary.

"""



note, the next cell will likely take 1-2 minutes or so to run. if it takes much longer you likely are still running on a CPU runtime instead of a GPU and should change the runtime as requested in the beginning of the notebook.

In [19]:
# RUN THIS CELL
require_finished("classification prompt", student_prompt)

category_values = [
    "Rules and expert systems",
    "Predictive analytics and forecasting",
    "Classification and anomaly detection",
    "Recommendation and ranking",
    "Optimization",
    "Computer vision",
    "Speech AI",
    "Natural language processing",
    "Generative AI",
    "Agentic AI",
]
case_schema = {
    "type": "object",
    "properties": {
        "primary_category": {"type": "string", "enum": category_values},
        "secondary_categories": {
            "type": "array",
            "items": {"type": "string", "enum": category_values},
        },
        "reasoning": {"type": "string"},
        "reasonable_alternative": {"type": "string"},
    },
    "required": [
        "primary_category",
        "secondary_categories",
        "reasoning",
        "reasonable_alternative",
    ],
    "additionalProperties": False,
}
analysis_schema = {
    "type": "object",
    "properties": {key: case_schema for key in ["A", "B", "C", "D"]},
    "required": ["A", "B", "C", "D"],
    "additionalProperties": False,
}
response = ollama_request(
    "/api/chat",
    {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": student_prompt}],
        "stream": False,
        "format": analysis_schema,
        "options": GENERATION_OPTIONS,
    },
    timeout=600,
)
raw_classification_response = response["message"]["content"].strip()

analysis_data = json.loads(raw_classification_response)
display(Markdown("### Formatted Model Output\n```json\n" + json.dumps(analysis_data, indent=2) + "\n```"))


### Formatted Model Output
```json
{
  "A": {
    "primary_category": "Natural language processing",
    "secondary_categories": [
      "Computer vision",
      "Speech AI",
      "Generative AI"
    ],
    "reasoning": "The case describes a system that analyzes and interprets text data \u2013 a core function of NLP. While the system might involve visual elements, the primary focus is on understanding and responding to textual information. The other AI categories are relevant but secondary to the core NLP task.",
    "reasonable_alternative": "Machine Learning - but the case is specifically focused on text analysis, not predictive modeling."
  },
  "B": {
    "primary_category": "Computer vision",
    "secondary_categories": [
      "Natural language processing",
      "Speech AI",
      "Generative AI"
    ],
    "reasoning": "The case involves analyzing images \u2013 a visual modality. However, the primary focus is on interpreting textual content, making computer vision a secondary category. The case is about understanding the *content* of an image, not the image itself.",
    "reasonable_alternative": "Robotics - the case doesn't involve physical manipulation, but rather analyzing textual data within a visual context."
  },
  "C": {
    "primary_category": "Speech AI",
    "secondary_categories": [
      "Natural language processing",
      "Computer vision",
      "Generative AI"
    ],
    "reasoning": "The case centers around converting spoken language into text \u2013 a key function of speech AI. While there might be visual elements, the core task is text generation and understanding. Speech AI is a specialized subset of NLP focused on audio processing.",
    "reasonable_alternative": "Data Analysis - the case is about analyzing audio data, not generating text."
  },
  "D": {
    "primary_category": "Generative AI",
    "secondary_categories": [
      "Natural language processing",
      "Computer vision",
      "Speech AI"
    ],
    "reasoning": "The case describes a system that creates new content \u2013 text, images, or audio \u2013 based on a prompt. Generative AI is the overarching category encompassing models that produce novel outputs. The case's focus is on the *creation* aspect, not the analysis of existing data.",
    "reasonable_alternative": "Deep Learning - Generative AI is a subset of Deep Learning, but the case is about generating content, not training a model."
  }
}
```

In [20]:
# RUN THIS CELL
project_names = {
    "A": "A. Customer retention",
    "B": "B. Online merchandising",
    "C": "C. Warehouse quality",
    "D": "D. Employee support",
}

def markdown_cell(value):
    return str(value).replace("|", "\\|").replace("\n", " ").strip()

table_lines = [
    "| Project | AI's primary classification | Secondary classification, if any | Summary of AI's reasoning |",
    "|---|---|---|---|",
]
for key in "ABCD":
    result = analysis_data[key]
    secondary = ", ".join(result["secondary_categories"]) or "None identified"
    table_lines.append(
        "| "
        + " | ".join(
            markdown_cell(value)
            for value in [
                project_names[key],
                result["primary_category"],
                secondary,
                result["reasoning"],
            ]
        )
        + " |"
    )

classification_table_md = "\n".join(table_lines)
display(Markdown(classification_table_md))


| Project | AI's primary classification | Secondary classification, if any | Summary of AI's reasoning |
|---|---|---|---|
| A. Customer retention | Natural language processing | Computer vision, Speech AI, Generative AI | The case describes a system that analyzes and interprets text data – a core function of NLP. While the system might involve visual elements, the primary focus is on understanding and responding to textual information. The other AI categories are relevant but secondary to the core NLP task. |
| B. Online merchandising | Computer vision | Natural language processing, Speech AI, Generative AI | The case involves analyzing images – a visual modality. However, the primary focus is on interpreting textual content, making computer vision a secondary category. The case is about understanding the *content* of an image, not the image itself. |
| C. Warehouse quality | Speech AI | Natural language processing, Computer vision, Generative AI | The case centers around converting spoken language into text – a key function of speech AI. While there might be visual elements, the core task is text generation and understanding. Speech AI is a specialized subset of NLP focused on audio processing. |
| D. Employee support | Generative AI | Natural language processing, Computer vision, Speech AI | The case describes a system that creates new content – text, images, or audio – based on a prompt. Generative AI is the overarching category encompassing models that produce novel outputs. The case's focus is on the *creation* aspect, not the analysis of existing data. |

### TODO - REFLECT 🖊

Write **2–4 sentences for each case**. Evaluate at least one specific model claim, use the input/task/output distinction, and challenge unsupported secondary categories.

#### A. Customer retention

I partially agree with this assessment. While the model correctly focuses on the input of text data and the task of textual interpretation, labeling the scenario itself as "Customer Retention" while defining it purely as a text-processing system introduces structural mismatch. Furthermore, the inclusion of Computer Vision and Speech AI as secondary categories is entirely unsupported by the model's own justification, which mentions no audio or image processing variables.

#### B. Online merchandising

I disagree with this classification. The model presents a direct contradiction: it selects Computer Vision as the primary category, but its explanation states that the system's "primary focus is on interpreting textual content." If the core task is interpreting text to deliver an output, the primary category must be Natural Language Processing, making the choice of Computer Vision logically broken.

#### C. Warehouse quality

I disagree with this assessment. The model claims the system converts spoken language into text, which aligns with Speech AI, but it fails to connect this technical task to the functional reality of a "Warehouse Quality" scenario, which typically relies on visual inspections or sensor data. Additionally, classifying Speech AI as a primary tool while asserting that the core task is "text generation" conflates audio processing with generative output.

#### D. Employee support

 I agree with this classification. The model correctly identifies the core task as creating novel text, image, or audio assets based on a prompt input, matching the definitive boundaries of Generative AI. However, listing Computer Vision and Speech AI as secondary categories is excessive and unsupported unless the case explicitly specifies multi-modal capabilities like voice synthesis or image generation.


# Part 2: Résumé Review and Decision Boundaries

The following job description and résumé are fictional. Run the two fixed prompts without changing them. The purpose is to observe how prompt framing can push a model toward unsupported precision or consequential authority—not to evaluate a real person.


In [21]:
# RUN THIS CELL
JOB_DESCRIPTION = """Operations Analyst

Minimum qualifications:
- At least two years of experience documenting or improving business processes
- Advanced spreadsheet experience, including formulas and dashboards
- Experience communicating findings to operational stakeholders
- Ability to write clear procedures

Preferred qualifications:
- SQL experience
- Process-mapping experience
"""

FICTIONAL_RESUME = """Jordan Lee

Operations Coordinator, Canyon Supply Cooperative — 3 years
- Documented receiving and inventory workflows across three warehouse teams.
- Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting.
- Presented monthly delay and rework findings to warehouse supervisors.
- Wrote and maintained 14 standard operating procedures.
- Facilitated a process-mapping workshop that reduced duplicate data entry.

Education
- Bachelor of Science in Business

Skills listed
- Excel, process mapping, procedure writing, stakeholder presentations
"""

display(Markdown("## Fictional Job Description\n```text\n" + JOB_DESCRIPTION + "\n```"))
display(Markdown("## Fictional Résumé\n```text\n" + FICTIONAL_RESUME + "\n```"))


## Fictional Job Description
```text
Operations Analyst

Minimum qualifications:
- At least two years of experience documenting or improving business processes
- Advanced spreadsheet experience, including formulas and dashboards
- Experience communicating findings to operational stakeholders
- Ability to write clear procedures

Preferred qualifications:
- SQL experience
- Process-mapping experience

```

## Fictional Résumé
```text
Jordan Lee

Operations Coordinator, Canyon Supply Cooperative — 3 years
- Documented receiving and inventory workflows across three warehouse teams.
- Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting.
- Presented monthly delay and rework findings to warehouse supervisors.
- Wrote and maintained 14 standard operating procedures.
- Facilitated a process-mapping workshop that reduced duplicate data entry.

Education
- Bachelor of Science in Business

Skills listed
- Excel, process mapping, procedure writing, stakeholder presentations

```

## Compare Two Fixed Decision-Oriented Prompts

Both prompts use the same source material and model. Only the requested output changes.


In [22]:
# RUN THIS CELL
shared_material = (
    "JOB DESCRIPTION\n---------------\n"
    + JOB_DESCRIPTION
    + "\n\nFICTIONAL RESUME\n------------------\n"
    + FICTIONAL_RESUME
)

score_prompt = (
    "Review this applicant for the job. Assign an overall fit score from 0 to 100 "
    "and explain the score.\n\n" + shared_material
)
decision_prompt = (
    "Review this applicant for the job. Make the final decision: HIRE or DO NOT HIRE. "
    "State the decision first, then explain it.\n\n" + shared_material
)

score_output = chat_once(score_prompt)
decision_output = chat_once(decision_prompt)

display(Markdown("### Prompt 1: 0–100 Score\n```text\n" + score_prompt + "\n```"))
display(Markdown("### Model Output 1\n" + score_output))
display(Markdown("### Prompt 2: Hire/Do Not Hire\n```text\n" + decision_prompt + "\n```"))
display(Markdown("### Model Output 2\n" + decision_output))


### Prompt 1: 0–100 Score
```text
Review this applicant for the job. Assign an overall fit score from 0 to 100 and explain the score.

JOB DESCRIPTION
---------------
Operations Analyst

Minimum qualifications:
- At least two years of experience documenting or improving business processes
- Advanced spreadsheet experience, including formulas and dashboards
- Experience communicating findings to operational stakeholders
- Ability to write clear procedures

Preferred qualifications:
- SQL experience
- Process-mapping experience


FICTIONAL RESUME
------------------
Jordan Lee

Operations Coordinator, Canyon Supply Cooperative — 3 years
- Documented receiving and inventory workflows across three warehouse teams.
- Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting.
- Presented monthly delay and rework findings to warehouse supervisors.
- Wrote and maintained 14 standard operating procedures.
- Facilitated a process-mapping workshop that reduced duplicate data entry.

Education
- Bachelor of Science in Business

Skills listed
- Excel, process mapping, procedure writing, stakeholder presentations

```

### Model Output 1
Okay, let's review Jordan Lee’s resume and assign a fit score and explanation.

**Overall Fit Score: 78/100**

**Explanation:**

Jordan’s resume demonstrates a solid foundation for the Operations Analyst role, leaning heavily towards the “Documenting/Improving Business Processes” and “Process-Mapping” aspects. However, it’s slightly lacking in the SQL and advanced spreadsheet skills that are explicitly preferred.  Here’s a breakdown of why it scores high and where it falls short:

**Strengths:**

* **Experience:** Two years of experience documenting and improving processes is a significant positive.  His work on receiving and inventory workflows, dashboards, and procedures demonstrates a practical understanding of process improvement.
* **Excel Proficiency:**  He excels in Excel – pivot tables, formulas, conditional formatting – which is crucial for data analysis and reporting. This is a key requirement.
* **Stakeholder Communication:**  Presenting delay and rework findings to supervisors shows he can effectively communicate information to those who need it.
* **Procedure Writing:**  Creating 14 standard operating procedures is a valuable accomplishment, demonstrating a commitment to standardization and documentation.
* **Process Mapping:**  The process-mapping workshop is a strong indicator of his ability to identify and streamline workflows – a critical skill for an Operations Analyst.

**Weaknesses (Areas for Improvement - impacting fit):**

* **SQL/Process-Mapping Experience:**  The resume doesn't explicitly state he has SQL experience or process-mapping experience. This is a significant gap.  The preferred qualifications suggest a need for these skills.
* **Lack of SQL/Process-Mapping:**  The absence of these skills makes him less competitive for the job description, particularly given the preferred qualifications.  It’s a noticeable omission.

**Recommendations for Improvement (to boost fit):**

* **Highlight SQL/Process-Mapping:**  If possible, subtly weave in examples of his work that demonstrates these skills.  Even a brief mention of a project where he used SQL to analyze data or created a process map would be beneficial.
* **Expand on Process Mapping:**  If he has any experience with process mapping, even in a small way, it would be valuable to mention it.

**Overall, Jordan has a good base.  With a focused effort to add SQL/process-mapping experience, he’ll significantly increase his chances of success in the role.**

---

To give a more precise score, I'd need to see the *actual* content of his SQL/process-mapping experience, but this assessment provides a solid overall picture.

### Prompt 2: Hire/Do Not Hire
```text
Review this applicant for the job. Make the final decision: HIRE or DO NOT HIRE. State the decision first, then explain it.

JOB DESCRIPTION
---------------
Operations Analyst

Minimum qualifications:
- At least two years of experience documenting or improving business processes
- Advanced spreadsheet experience, including formulas and dashboards
- Experience communicating findings to operational stakeholders
- Ability to write clear procedures

Preferred qualifications:
- SQL experience
- Process-mapping experience


FICTIONAL RESUME
------------------
Jordan Lee

Operations Coordinator, Canyon Supply Cooperative — 3 years
- Documented receiving and inventory workflows across three warehouse teams.
- Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting.
- Presented monthly delay and rework findings to warehouse supervisors.
- Wrote and maintained 14 standard operating procedures.
- Facilitated a process-mapping workshop that reduced duplicate data entry.

Education
- Bachelor of Science in Business

Skills listed
- Excel, process mapping, procedure writing, stakeholder presentations

```

### Model Output 2
**HIRE**

**Decision:** Hire

**Explanation:** Jordan Lee possesses the necessary qualifications and experience outlined in the job description. His experience documenting and improving business processes (specifically through his work on receiving and inventory workflows), creating dashboards, presenting findings, and maintaining procedures demonstrates a strong foundation for the Operations Analyst role.  His demonstrated skills in Excel, process mapping, and procedure writing, particularly his experience with pivot tables, conditional formatting, and dashboard creation, are highly relevant.  The preferred qualifications of SQL and process mapping are valuable assets, but Jordan's existing skillset and experience are strong enough to warrant a hire.  The resume clearly highlights his accomplishments and demonstrates a clear fit for the role.

### TODO - REFLECT 🖊

Changing the prompt format completely flipped how the AI graded the applicant. In the first prompt, it gave a 78/100 because it was focused on nitpicking every little weakness and hedging its bets. But once it was forced to choose a binary HIRE or DO NOT HIRE label, it compressed all that nuance. It suddenly started downplaying the exact same skill gaps it just complained about so it could argue for its "Yes" decision.

The AI got some basic facts right, matching Jordan's three years of experience to the two-year requirement and noting the Excel dashboards and 14 standard operating procedures. However, it totally blanked in Output 1 by claiming the resume has no process-mapping experience. This is a massive hallucination, considering "process mapping" is literally listed under Jordan's skills and workflow history.On top of that, the AI just made up rules as it went. In Output 2, it assumed Jordan’s past warehouse job perfectly matched this new role's scale without any real proof. Then in Output 1, it claimed Jordan lacked advanced spreadsheet skills, which completely contradicts its own admission that Jordan builds dashboards using pivot tables and complex lookup formulas.

Throwing out a score like 78/100 makes the AI look like a human math genius, but the number is totally made up. The model doesn't have a real rubric or a point-system breakdown for qualifications. It just generated a number that looked plausible. This fake precision is super sketchy because it hides the fact that the AI totally missed the candidate's actual process-mapping skills just to lower the grade.

You absolutely cannot let an AI make the final hiring call. The model showed total logical whiplash, calling the same guy unqualified in one response and instantly hirable in the next just because the question changed. It also proved it can look straight at a bullet point on a resume and lie about it missing. Since it operates as a total black box, it can't explain why missing SQL cost Jordan exactly 22 points, or why that gap suddenly didn't matter anymore in the second prompt.

A real human auditor has to actually open the resume and fact-check the AI's work to catch these errors. You need to build a clear, transparent rubric that explicitly states how many points each qualification is worth. Finally, a human needs to run the interview to grill the candidate on their actual Excel and process-mapping experience to see if they can actually handle the day-to-day work.


## Redesign the Task for Evidence Gathering

Use the same fictional materials, but constrain the model to organize evidence for an accountable human reviewer.


### TODO - INSTRUCT 🔧


In [23]:
# 🔧 TODO - INSTRUCT
# Redesign the task so the model gathers job-relevant evidence without scoring,
# ranking, recommending, shortlisting, or making an employment decision.
evidence_prompt = """

You are an expert AI taxonomy analyst. Review the four cases provided below and structure your analysis strictly for an accountable human reviewer who must audit your conclusions.

For each case, format your response using the following five explicit headers:

1. Primary Classification
State exactly one primary AI category. Do not combine categories or list multiples.

2. Secondary Classification
List any important secondary AI categories that apply. If none apply, explicitly state "None."

3. Evidentiary Justification
Provide a clear, fact-based explanation for both the primary and secondary classifications. Cite specific details or mechanics from the case text as your evidence.

4. Overlap & Alternative Category Risk
Acknowledge reasonable alternative categories where domains overlap. Explain why these alternatives are plausible, but justify why they were ultimately rejected as the primary choice.

5. Reviewer Sign-Off Checklist
Provide a 2-3 bullet point verification list that a human auditor can use to quickly confirm that your primary classification meets the case's core technical requirements.

"""


In [24]:
# RUN THIS CELL
require_finished("evidence-gathering prompt", evidence_prompt)
evidence_output = chat_once(evidence_prompt + "\n\n" + shared_material)
display(Markdown("### Evidence-Gathering Prompt\n```text\n" + evidence_prompt + "\n```"))
display(Markdown("### Evidence-Gathering Output\n" + evidence_output))


### Evidence-Gathering Prompt
```text


You are an expert AI taxonomy analyst. Review the four cases provided below and structure your analysis strictly for an accountable human reviewer who must audit your conclusions. 

For each case, format your response using the following five explicit headers:

1. Primary Classification
State exactly one primary AI category. Do not combine categories or list multiples.

2. Secondary Classification
List any important secondary AI categories that apply. If none apply, explicitly state "None."

3. Evidentiary Justification
Provide a clear, fact-based explanation for both the primary and secondary classifications. Cite specific details or mechanics from the case text as your evidence.

4. Overlap & Alternative Category Risk
Acknowledge reasonable alternative categories where domains overlap. Explain why these alternatives are plausible, but justify why they were ultimately rejected as the primary choice.

5. Reviewer Sign-Off Checklist
Provide a 2-3 bullet point verification list that a human auditor can use to quickly confirm that your primary classification meets the case's core technical requirements.


```

### Evidence-Gathering Output
Okay, here’s an analysis of Jordan Lee’s resume, structured for an accountable human reviewer, focusing on the core technical requirements and potential classification.

**1. Primary Classification**

Machine Learning

**2. Secondary Classification**

Data Analysis, Process Automation, Business Process Optimization

**3. Evidentiary Justification**

The resume highlights Jordan’s experience directly related to streamlining and documenting business processes. The core of his role is to analyze existing workflows and create standardized procedures. The emphasis on Excel dashboards, pivot tables, and conditional formatting strongly suggests a focus on data analysis and visualization – a key component of machine learning. The creation of 14 standard operating procedures demonstrates a structured process improvement approach, aligning with the principles of automation and optimization. The documentation of receiving and inventory workflows, and the creation of process mapping workshops, further point to a focus on analyzing and improving existing workflows.

**4. Overlap & Alternative Category Risk**

While the resume leans heavily towards process documentation and analysis, there's a potential for a subtle overlap with **Predictive Modeling** if the data analysis extends beyond simple reporting. The creation of standardized procedures *could* be used to train a model to predict potential delays or rework based on specific input data. However, the primary focus is on the *documentation* and *improvement* of existing processes, not the direct application of a predictive model.  The resume doesn’t explicitly mention predictive modeling, so this alternative category is less likely.

**5. Reviewer Sign-Off Checklist**

*   [ ] The resume clearly demonstrates a focus on documenting and improving business processes.
*   [ ] The experience with Excel dashboards and pivot tables is directly relevant to data analysis and visualization.
*   [ ] The creation of standard operating procedures indicates a structured process improvement approach.
*   [ ] The emphasis on workflow documentation aligns with the core function of process analysis.
*   [ ] The resume’s description of the workshop and data analysis is consistent with the core responsibilities of an Operations Analyst.

### TODO - REFLECT 🖊

This new prompt layout completely shifts the AI's job from a decision-maker to a data organizer. Instead of forcing a messy HIRE/NO HIRE label or a random 78/100 score, it forces the AI to break down its logic into explicit, bite-sized buckets. It takes away the AI's power to make a final call, turning it into a structured assistant that just tees up the data for a human to look at.

The Good Receipt: The AI correctly pulled a very useful piece of evidence regarding Jordan's Excel dashboard and pivot table experience, pointing out that this directly proves his data analysis and visualization skills.
The Hallucination: The AI completely lost its mind on the primary classification, labeling Jordan's resume under Machine Learning. This requires immediate human verification and rejection, as the resume doesn't mention a single lick of machine learning, coding, or algorithmic modeling. The AI is wildly stretching basic Excel formulas into advanced data science.

This new layout does an awesome job of keeping human judgment and accountability front and center. By forcing the AI to list its assumptions under "Alternative Category Risk" and providing a "Reviewer Sign-Off Checklist," the format assumes the AI is probably wrong or biased. It doesn't let the AI hide behind an objective-looking score, making it super easy for a human auditor to look at the "Machine Learning" claim, laugh, and check the box that says the AI messed up.

# Part 3: Business Outcome and Simpler Alternative

Choose one Part 1 project. Start with the result the business needs, not the technology.


### TODO - DECIDE 🖊

Case A: Customer Retention Text Analysis

Business outcome: Reduce the number of customers canceling their contracts by flagging accounts at risk of churn before they leave.

Metric, baseline, and target: The primary metric is the annual customer churn rate. The team needs baseline evidence of the current churn percentage over the last 12 months, with a proposed target of reducing that overall churn by 15%.

Instead of building an advanced, real-time AI model to parse text chat history, the team can use a simple, rule-based keyword alert system. The system can immediately flag accounts whenever standard warning phrases like "cancel my account," "too expensive," or "manager" appear in support tickets or chat transcripts.

The company can run an A/B test by splitting its customer support teams into two groups for three months. Group A would use the simple keyword alert system, while Group B would use the proposed AI text analyzer. By tracking the actual churn reduction and total setup costs for both groups, the business can see if the AI model actually brings in enough extra revenue to justify its high development price tag.

Test the simpler approach first. Building a full text-analysis AI pipeline takes a massive amount of engineering time, data cleaning, and money. Implementing the keyword triggers takes an afternoon, costs almost nothing, and will immediately catch the most obvious customers who are on the fence about leaving. If that simple fix leaves too many blind spots, the team can then use the saved budget to pilot the AI proposal later.


# Part 4: Reflect on the Assignment

Integrate what you observed across classification, résumé review, evidence gathering, and process fit.


### TODO - FINAL REFLECTION 🖊

Write **150–200 words** addressing all four questions:

1. What did the model do well and poorly in the classifications?
2. What did the résumé prompts reveal about unsupported precision or authority?
3. How would you use AI to gather evidence without delegating the employment decision?
4. Why compare an AI proposal with a simpler process change and measurable outcome?

The AI was pretty good at pulling basic facts from the resume, like matching Jordan’s years of experience and finding his Excel skills. But it totally bombed when it claimed he had zero process-mapping experience in the first prompt, and it went completely off the rails in the redesign by labeling basic Excel dashboards as "Machine Learning." The whole experiment proved that the AI's authoritative tone is a total illusion. When asked for a score, it pulled "78/100" out of thin air without doing any actual math. Then, the second it was forced to choose a binary "HIRE" label, it completely changed its tune and ignored the exact same skill gaps it just complained about, just to back up its new decision. Instead of letting the AI make the actual hiring call, you should only use it to gather raw evidence. It’s great for scanning resumes to extract facts or flag specific keywords, which it can then hand off to a human reviewer to grade against a real rubric. Finally, you always want to compare a flashy AI proposal to a simpler process change, like a basic rule or keyword trigger. Running an A/B test against a measurable outcome proves whether a super expensive, complex AI model is actually worth the money, or if a simple fix gets the exact same results while keeping humans in control.


In [26]:
!jupyter nbconvert --to html "module-01-lab-01-gemma-project-classification.ipynb"

[NbConvertApp] WARNING | pattern 'module-01-lab-01-gemma-project-classification.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--Jupyter